In [ ]:
"""
     Cluster point cloud based on spatial gaps using DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
"""

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import os
import open3d as o3d
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ignore warning information
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def segment_point_cloud_by_gap(pts, eps=0.05, min_samples=10):
    """
    Args:
        pts (np.ndarray): Point cloud (N, 3)
        eps (float): Max distance between two samples for them to be in the same neighborhood
        min_samples (int): Min samples in a neighborhood to form a core point

    Returns:
        labels (np.ndarray): Cluster labels for each point (-1 = noise)
    """
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(pts)
    return labels

In [ ]:
# load point cloud
POINTCLOUD_PATH = './data/pointcloud/'
files = os.listdir(POINTCLOUD_PATH)
pointclouds = []

for file in files:
    if file.startswith("calibrated_pointcloud"):
        pointclouds.append(file)

pointcloud_file_path = os.path.join(POINTCLOUD_PATH, pointclouds[2])
pointcloud = pd.read_csv(pointcloud_file_path, delim_whitespace=True, header=None, names=['X', 'Y', 'Z'])
pointcloud = pointcloud.iloc[:, :3].to_numpy()

# preprocessing: downsample for faster computation (optional)
pc = o3d.geometry.PointCloud()
pc.points = o3d.utility.Vector3dVector(pointcloud)
pc = pc.voxel_down_sample(voxel_size=0.1)

pointcloud = np.asarray(pc.points)
print(">>> Data type of point cloud: ", type(pointcloud))
print(">>> Shape of point cloud: ", pointcloud.shape)

x = pointcloud[:, 0]
y = pointcloud[:, 1]
z = pointcloud[:, 2]

# re-define the boundary of cmap for better visualization
from matplotlib.colors import Normalize
from matplotlib import cm
cmap = cm.viridis
vmin = 0.0
vmax = 1.2
norm = Normalize(vmin=vmin, vmax=vmax)

plt.figure()
scatter = plt.scatter(y, -x, c=z, cmap=cmap, norm=norm, s=0.1)
plt.colorbar(scatter, label='z')  
plt.axis('off')
plt.show()

In [ ]:
# label to color function
def label2color(seg_labels):
    ''' map segementation labels to colors '''
    map_label_to_color = {
         0: [31, 119, 180],     # Blue
         1: [44, 160, 44],      # Green
         2: [255, 127, 14],     # Orange
         3: [214, 39, 40],      # Red
         4: [255, 187, 120],    # Light Orange
         5: [152, 223, 138],    # Light Green
         6: [174, 199, 232],    # Light Blue
         7: [255, 152, 150],    # Light Red / Salmon
         8: [148, 103, 189],    # Purple
         9: [197, 176, 213],    # Lavender / Light Purple
        10: [140, 86, 75],      # Brown
        11: [196, 156, 148],    # Light Brown / Tan
        12: [227, 119, 194],    # Pink
        13: [247, 182, 209],    # Light Pink / Rose
        14: [127, 127, 127],    # Gray
        15: [199, 199, 199],    # Light Gray
        16: [188, 189, 34],     # Olive / Yellow-Green
        17: [219, 219, 141],    # Light Olive / Khaki
        18: [23, 190, 207],     # Teal / Cyan
        19: [158, 218, 229],    # Light Cyan / Sky Blue
        -1: [255, 255, 255]     # white (ignore)
    }
    colors = np.array([map_label_to_color[label] for label in seg_labels])
    return colors

In [ ]:
labels = segment_point_cloud_by_gap(pointcloud, eps=0.26, min_samples=20)
print(">>> Cluster labels (with -1 = noise): ", set(labels))

In [ ]:
# visualize point cloud
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=pointcloud[:, 0], y=pointcloud[:, 1], z=pointcloud[:, 2], 
            mode='markers',
            marker=dict(size=1, color=label2color(labels))
        )
    ],
    layout=dict(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False, range=[-20, 20])
        )
    )
)

# set background colors
fig.update_layout(
    plot_bgcolor='lightgrey', 
    paper_bgcolor='lightgrey'
)

fig.show()